In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

uv add toyaikit

In [2]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [10]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

### Defining the tool
First we define a top-level search function that queries the index directly. The model will reference it by this name. We keep the Python function and the tool name aligned so the dispatch is easier later.

In [3]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

Next we tell the model about this function. The model doesn't see our Python code, only a schema describing what the function does and what arguments it takes. LLMs are language agnostic. At the end we're just making an HTTP call, so we describe the tool in JSON rather than in Python. The same schema would work from TypeScript or Java.

In [4]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

The description is the most important field, because the model reads it to decide when to call the function. parameters is a JSON schema for the arguments, and we mark query as required so the model always fills it in.

### Registering the tool

In [5]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

### Letting ToyAIKit generate the schema
Writing that schema by hand is annoying, and we don't want to do it for every function. So we don't have to.

If we add a type hint and a docstring to search, ToyAIKit reads them and derives the schema for us:

In [6]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

Then register it without passing a schema:

In [7]:
agent_tools = Tools()
agent_tools.add_tool(search)

You can look at what ToyAIKit produced.

In [8]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

### The chat interface and runner
Create the chat interface and a callback, then build the runner:

In [11]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-nano")
)

The chat_interface handles display in the notebook. The callback renders model messages and tool calls as they happen. The runner runs the agent loop, the same while True we wrote by hand. It sends messages, executes function calls, adds tool outputs back, and repeats until the model is done.

We pick gpt-5.4-mini here on purpose. Without it, ToyAIKit falls back to a smaller, faster default that doesn't follow the instructions as reliably.

### Running one prompt

In [12]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


We used the typo "Olama" on purpose. The agent searches and gets poor results, then retries with "Ollama". The recovery is the same as the handwritten loop. The notebook output is nicer to watch. Each tool call and message renders inline, so you can look at every search result.

The result is a LoopResult with all_messages (the full conversation), token counts, and cost (computed from token usage).

### Cost and tokens

In [13]:
result.cost

CostInfo(input_cost=Decimal('0.0001022'), output_cost=Decimal('0.00055'), total_cost=Decimal('0.0006522'))

You can also look at the full message history.

In [14]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Ollama run locally install"}', call_id='call_Bk7eeOiEBv2hoqwRsP49VzcJ', name='search', type='function_call', id='fc_088c514e85172958006a34be8ce9248192bb923b9e1813df7d', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"Ollama local run command pull model run"}', call_id='call_WIkqL5RQin2T

This is just a list - the same messages list we maintained by hand.

### Continuing the conversation
Take the messages from the previous result and pass them as previous_messages on the next loop call:

In [15]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


The runner picks up where the last call left off, with the same agent loop and an extended history. The model knows "different model" refers to Ollama because it sees the previous turn in memory. Without that history, it would have no idea what we're asking about.

### Interactive chat
For a chat-like workflow, run the built-in input loop:

In [ ]:
runner.run()

You: how long does this course takes?


-> Response received


-> Response received


You: llm


-> Response received


-> Response received
